# Label-free metric tuning

This notebook tunes six anomaly detectors on normal-only validation data using label-free metrics, then checks the selected configurations on a seeded holdout split with real anomalies.

In [ ]:
import warnings

import numpy as np
import optuna
import pandas as pd
from oddball import load
from pyod.models.iforest import IForest as PyodIForest
from pyod.models.knn import KNN
from pyod.models.lof import LOF as PyodLOF
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from labelfree.metrics import (
    asoi_score,
    bounding_box_volume,
    excess_mass_auc,
    expected_anomaly_gap_score,
    laplacian_score,
    mass_volume_auc,
    normalized_pseudo_discrepancy_score,
    relative_top_median_score,
    score_cluster_metrics,
    sireos_score,
)
from labelfree.utils.validation import orient_scores

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.ERROR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

SEED = 7
DATASET = "cardio"  # easy swap: "wbc", "breastw", "satimage2", ...
CONTAMINATION = 0.10
rng = np.random.default_rng(SEED)

In [ ]:
X, y = load(DATASET)
y = (y != 0).astype(int)
normal = X[y == 0]
anomaly = X[y == 1]

normal = normal[rng.choice(len(normal), min(300, len(normal)), replace=False)]
anomaly = anomaly[rng.choice(len(anomaly), min(80, len(anomaly)), replace=False)]

X_train, X_rest = train_test_split(normal, test_size=0.40, random_state=SEED)
X_tune, X_hold_normal = train_test_split(X_rest, test_size=0.50, random_state=SEED)
X_hold = np.vstack([X_hold_normal, anomaly])
y_hold = np.r_[
    np.zeros(len(X_hold_normal), dtype=int), np.ones(len(anomaly), dtype=int)
]

pd.DataFrame(
    [
        {
            "dataset": DATASET,
            "features": X.shape[1],
            "train_normals": len(X_train),
            "tune_normals": len(X_tune),
            "holdout_normals": len(X_hold_normal),
            "holdout_anomalies": len(anomaly),
        }
    ]
)

## Detector pool

scikit-learn detectors expose `decision_function` with larger values meaning more normal. PyOD detectors expose larger values as more anomalous. The metric calls receive that polarity explicitly.

In [ ]:
model_info = pd.DataFrame(
    [
        ("sklearn_iforest", "sklearn", "higher_is_normal"),
        ("sklearn_ocsvm", "sklearn", "higher_is_normal"),
        ("sklearn_lof", "sklearn", "higher_is_normal"),
        ("pyod_iforest", "pyod", "higher_is_anomalous"),
        ("pyod_knn", "pyod", "higher_is_anomalous"),
        ("pyod_lof", "pyod", "higher_is_anomalous"),
    ],
    columns=["model", "library", "score_polarity"],
)
model_info

## Label-free objectives

Each objective is computed only on normal tuning data, plus simple uniform reference samples where the metric requires them. Lower-is-better metrics are negated for Optuna.

In [ ]:
metric_info = pd.DataFrame(
    [
        ("rtm", "higher"),
        ("eag", "higher"),
        ("score_silhouette", "higher"),
        ("asoi", "higher"),
        ("excess_mass", "higher"),
        ("npd", "higher"),
        ("laplacian", "lower"),
        ("sireos", "lower"),
        ("mass_volume", "lower"),
    ],
    columns=["metric", "better"],
)
metric_info

## Optuna tuning

The objective below instantiates the selected detector, fits only normal training data, scores normal tuning data, computes the practical metric set below, and gives Optuna only the selected objective value. Exact IREOS is excluded because its repeated kernel optimization is intentionally expensive.

In [ ]:
def label_free_metric_values(X_scaled, scores, reference_scores, polarity):
    support_volume = bounding_box_volume(X_scaled)
    clusters = score_cluster_metrics(
        scores, contamination=CONTAMINATION, score_polarity=polarity
    )
    return {
        "rtm": relative_top_median_score(scores, score_polarity=polarity),
        "eag": expected_anomaly_gap_score(scores, score_polarity=polarity),
        "score_silhouette": clusters["silhouette"],
        "asoi": asoi_score(
            X_scaled, scores, contamination=CONTAMINATION, score_polarity=polarity
        ),
        "excess_mass": excess_mass_auc(
            scores,
            reference_scores,
            support_volume=support_volume,
            level_count=80,
            score_polarity=polarity,
        ),
        "npd": normalized_pseudo_discrepancy_score(
            scores, reference_scores, score_polarity=polarity
        ),
        "laplacian": laplacian_score(
            X_scaled, scores, n_neighbors=10, score_polarity=polarity
        ),
        "sireos": sireos_score(X_scaled, scores, score_polarity=polarity),
        "mass_volume": mass_volume_auc(
            scores,
            reference_scores,
            support_volume=support_volume,
            alpha_min=0.80,
            alpha_max=0.98,
            alpha_count=80,
            score_polarity=polarity,
        ),
    }


rows = []

for metric_name, metric_better in metric_info.to_records(index=False):
    for model_name, library, polarity in model_info.to_records(index=False):

        def objective(trial):
            if model_name == "sklearn_iforest":
                model = IsolationForest(
                    n_estimators=trial.suggest_int("n_estimators", 50, 160),
                    max_samples=trial.suggest_int("max_samples", 32, len(X_train)),
                    contamination=CONTAMINATION,
                    random_state=SEED,
                    n_jobs=-1,
                )
            elif model_name == "sklearn_ocsvm":
                model = OneClassSVM(
                    nu=trial.suggest_float("nu", 0.02, 0.20),
                    gamma=trial.suggest_float("gamma", 1e-3, 1.0, log=True),
                )
            elif model_name == "sklearn_lof":
                model = LocalOutlierFactor(
                    n_neighbors=trial.suggest_int("n_neighbors", 5, 40),
                    contamination=CONTAMINATION,
                    novelty=True,
                )
            elif model_name == "pyod_iforest":
                model = PyodIForest(
                    n_estimators=trial.suggest_int("n_estimators", 50, 160),
                    max_samples=trial.suggest_int("max_samples", 32, len(X_train)),
                    contamination=CONTAMINATION,
                    random_state=SEED,
                )
            elif model_name == "pyod_knn":
                model = KNN(
                    n_neighbors=trial.suggest_int("n_neighbors", 5, 40),
                    contamination=CONTAMINATION,
                )
            else:
                model = PyodLOF(
                    n_neighbors=trial.suggest_int("n_neighbors", 5, 40),
                    contamination=CONTAMINATION,
                )

            pipe = make_pipeline(StandardScaler(), model)
            pipe.fit(X_train)
            tune_scores = pipe.decision_function(X_tune)
            scaler = pipe.named_steps["standardscaler"]
            X_tune_scaled = scaler.transform(X_tune)
            low = X_tune_scaled.min(axis=0)
            high = X_tune_scaled.max(axis=0)
            reference = np.random.default_rng(SEED + trial.number).uniform(
                low, high, size=(300, X_tune_scaled.shape[1])
            )
            reference_scores = pipe.named_steps[
                list(pipe.named_steps)[-1]
            ].decision_function(reference)

            metric_values = label_free_metric_values(
                X_tune_scaled, tune_scores, reference_scores, polarity
            )
            trial.set_user_attr("label_free_metrics", metric_values)
            value = metric_values[metric_name]

            return -value if metric_better == "lower" else value

        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=6, show_progress_bar=False)
        best_metrics = study.best_trial.user_attrs["label_free_metrics"]
        rows.append(
            {
                "metric": metric_name,
                "better": metric_better,
                "model": model_name,
                "library": library,
                "score_polarity": polarity,
                "objective": study.best_value,
                "metric_value": best_metrics[metric_name],
                "params": study.best_params,
                **{
                    f"tune_{name}": best_metrics[name] for name in metric_info["metric"]
                },
            }
        )

leaderboard = pd.DataFrame(rows)
leaderboard.sort_values(["metric", "objective"], ascending=[True, False]).groupby(
    "metric"
).head(1)

## Best configuration per metric

This is the model/configuration each label-free metric would pick on the normal-only tuning split.

In [ ]:
best_by_metric = (
    leaderboard.sort_values(["metric", "objective"], ascending=[True, False])
    .groupby("metric", as_index=False)
    .head(1)
    .sort_values("metric")
    .reset_index(drop=True)
)
best_by_metric[["metric", "better", "model", "library", "metric_value", "params"]]

## Holdout check with labels

Labels are used only here. ROC AUC and PR AUC use oriented anomaly scores. F1 is computed at the true anomaly count in the holdout split so every selected configuration predicts the same number of anomalies.

In [ ]:
results = []
X_fit = np.vstack([X_train, X_tune])
k = int(y_hold.sum())

for row in best_by_metric.to_dict("records"):
    params = row["params"]
    model_name = row["model"]
    polarity = row["score_polarity"]

    if model_name == "sklearn_iforest":
        model = IsolationForest(
            contamination=CONTAMINATION, random_state=SEED, n_jobs=-1, **params
        )
    elif model_name == "sklearn_ocsvm":
        model = OneClassSVM(**params)
    elif model_name == "sklearn_lof":
        model = LocalOutlierFactor(contamination=CONTAMINATION, novelty=True, **params)
    elif model_name == "pyod_iforest":
        model = PyodIForest(contamination=CONTAMINATION, random_state=SEED, **params)
    elif model_name == "pyod_knn":
        model = KNN(contamination=CONTAMINATION, **params)
    else:
        model = PyodLOF(contamination=CONTAMINATION, **params)

    pipe = make_pipeline(StandardScaler(), model)
    pipe.fit(X_fit)
    raw_scores = pipe.decision_function(X_hold)
    anomaly_scores = orient_scores(raw_scores, score_polarity=polarity)
    y_pred = np.zeros_like(y_hold)
    y_pred[np.argsort(anomaly_scores)[-k:]] = 1

    result = {
        "selected_by": row["metric"],
        "model": model_name,
        "roc_auc": roc_auc_score(y_hold, anomaly_scores),
        "pr_auc": average_precision_score(y_hold, anomaly_scores),
        "f1_at_k": f1_score(y_hold, y_pred),
    }
    result.update(
        {f"tune_{name}": row[f"tune_{name}"] for name in metric_info["metric"]}
    )
    results.append(result)

holdout_results = pd.DataFrame(results)
holdout_results.sort_values(["roc_auc", "pr_auc"], ascending=False).reset_index(
    drop=True
)

## Label-free and labeled metric agreement

In [ ]:
tune_columns = [f"tune_{name}" for name in metric_info["metric"]]
holdout_results[tune_columns + ["roc_auc", "pr_auc", "f1_at_k"]].corr().loc[
    tune_columns, ["roc_auc", "pr_auc", "f1_at_k"]
]